# Parquetize the sample seed

Dev tool. Turns the committed JSONL **sample seed** under `data/bootstrap/` into
a Parquet corpus the store can read.

The seed gets its **own** corpus folder, `data/bootstrap/lexicon/`, kept separate
from the real ingestion build under `data/lexicon/` so the two never overwrite
each other. The whole `data/` tree is gitignored, so the generated Parquet (and
the `_store.sqlite` cache the store builds beside it) stay uncommitted; the
committed seed JSONL stays the editable source.

Thin caller: every operation is `corpus.import_table`, which validates each row
through its model before writing the canonical Parquet. Requires the `store`
extra only for the `inspect_table` check at the end (`uv sync --extra store`);
`pyarrow` itself is a base dependency.


In [ ]:
from lang_tools.lexicon.corpus import import_table
from lang_tools.lexicon.corpus import inspect_table
from lang_tools.lexicon.lemma_store import TABLES
from lang_tools.params.lang_tools_params import get_lang_tools_params

data_fol = get_lang_tools_params().paths.data_fol
# The seed folder is both the JSONL source AND the seed corpus root, so the
# parquet lands at data/bootstrap/lexicon/ (not the real data/lexicon/).
seed_fol = data_fol / "bootstrap"
seed_fol

## Build `data/bootstrap/lexicon/` from the seed

One `import_table` per table. `lemmas` / `senses` land partitioned per language.


In [ ]:
for name in TABLES:
    rows = import_table(name, seed_fol / f"{name}.jsonl", data_fol=seed_fol)
    print(f"{name}: {rows} rows -> data/bootstrap/lexicon")

## Verify

Read a couple of tables back through DuckDB to confirm the Parquet is there.


In [ ]:
inspect_table("concepts", data_fol=seed_fol)

In [ ]:
inspect_table("lemmas", data_fol=seed_fol, lang="pt", limit=5)

In [ ]:
inspect_table("senses", data_fol=seed_fol, limit=5)